# Salary models

Цель: обучить модели для прогнозирования зарплаты (в лог‑шкале) по:
1) табличным признакам (компетенции/регион/год) — CatBoost
2) тексту вакансии — TF‑IDF + Ridge (OOF прогнозы)

**Вход:**
- `vacancies_top30_competencies_train_ready.csv` (табличный датасет)
- `vacancies_ml_related_with_salary.csv` (датасет с текстом и зарплатой)

**Выход:**
- `catboost_salary_log_model.cbm`
- `catboost_salary_log_feature_importance.csv`
- `catboost_salary_log_test_predictions.csv`
- `vacancies_text_preprocessed.csv`
- `vacancies_text_model_oof_predictions.csv`

In [2]:
from __future__ import annotations

import html
import re
import unicodedata

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

from catboost import CatBoostRegressor

In [3]:
# =========================
# Config
# =========================
from pathlib import Path

RANDOM_STATE = 42

# Paths (robust to running from any CWD)
# Prefer: <repo>/notebooks/salary_models.ipynb -> PROJECT_ROOT = <repo>
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd if (_cwd / "artifacts").exists() else _cwd.parent
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
DATA_DIR = PROJECT_ROOT / "data"

TABULAR_FILE = ARTIFACTS_DIR / "vacancies_train_ready.csv"
TEXT_FILE = DATA_DIR / "vacancies_ml_related_with_salary.csv"

# Fail fast with readable error (instead of pandas FileNotFoundError later)
if not TABULAR_FILE.exists():
    raise FileNotFoundError(f"TABULAR_FILE not found: {TABULAR_FILE}")
if not TEXT_FILE.exists():
    raise FileNotFoundError(f"TEXT_FILE not found: {TEXT_FILE}")

CATBOOST_MODEL_OUT = ARTIFACTS_DIR / "catboost_salary_log_model.cbm"
CATBOOST_FI_OUT = ARTIFACTS_DIR / "catboost_salary_log_feature_importance.csv"
CATBOOST_TEST_PRED_OUT = ARTIFACTS_DIR / "catboost_salary_log_test_predictions.csv"

TEXT_PREPROCESSED_OUT = ARTIFACTS_DIR / "vacancies_text_preprocessed.csv"
TEXT_OOF_OUT = ARTIFACTS_DIR / "vacancies_text_model_oof_predictions.csv"

# Currency rates -> RUB (adjust if needed)
CURRENCY_RATES_TO_RUB = {
    "RUB": 1.0,
    "RUR": 1.0,
    "USD": 92.0,
    "EUR": 100.0,
    "KZT": 0.20,
    "BYN": 28.0,
    "UAH": 2.2,
    "UZS": 0.0073,
    "GEL": 34.0,
    "KGS": 1.05,
    "AZN": 54.0,
}

In [4]:
# =========================
# Utils: salary parsing
# =========================

def detect_currency(text: object) -> str | None:
    if pd.isna(text):
        return None

    s = str(text).lower()

    if any(x in s for x in ["руб", "rur", "rub", "₽"]):
        return "RUB"
    if any(x in s for x in ["usd", "$", "доллар"]):
        return "USD"
    if any(x in s for x in ["eur", "€", "евро"]):
        return "EUR"
    if any(x in s for x in ["kzt", "₸", "тенге"]):
        return "KZT"
    if any(x in s for x in ["byn", "бел. руб", "белорус"]):
        return "BYN"
    if any(x in s for x in ["uah", "грн"]):
        return "UAH"
    if any(x in s for x in ["uzs", "сум"]):
        return "UZS"
    if any(x in s for x in ["gel", "лари"]):
        return "GEL"
    if any(x in s for x in ["kgs", "сом"]):
        return "KGS"
    if any(x in s for x in ["azn", "манат"]):
        return "AZN"

    return None


def parse_salary_to_rub(raw_value: object) -> float:
    if pd.isna(raw_value):
        return float("nan")

    s = str(raw_value).strip().lower()
    if s in {"", "none", "null", "nan", "не указана"}:
        return float("nan")

    currency = detect_currency(s)

    # remove separators inside numbers: "150 000" -> "150000"
    s = re.sub(r"(?<=\d)\s+(?=\d)", "", s)

    numbers = re.findall(r"\d+(?:[.,]\d+)?", s)
    values = [float(x.replace(",", ".")) for x in numbers]

    if not values:
        return float("nan")

    salary_value = values[0] if len(values) == 1 else (min(values[0], values[1]) + max(values[0], values[1])) / 2
    rate = CURRENCY_RATES_TO_RUB.get(currency or "", float("nan"))

    return salary_value * rate

In [ ]:
# =========================
# Part A: Tabular model (CatBoost)
# =========================
df_tab = pd.read_csv(str(TABULAR_FILE))

TARGET_COL = "salary_rub_log"
if TARGET_COL not in df_tab.columns:
    raise KeyError(f"Target column `{TARGET_COL}` not found in {TABULAR_FILE}")

X = df_tab.drop(columns=[TARGET_COL])
y = df_tab[TARGET_COL].copy()

# Avoid leakage if present
X = X.drop(columns=["salary_rub"], errors="ignore")

# Dataset schema (artifacts/vacancies_train_ready.csv) contains identifiers + URL + several text group columns
DROP_COLS = ["vacancy_id", "vacancy_url"]
X = X.drop(columns=[c for c in DROP_COLS if c in X.columns], errors="ignore")

# Columns with dtype=object in this dataset (group fields). Treat them as categorical.
cat_features = [
    c for c in X.columns
    if c.startswith("skill_")
]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.15, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "Valid:", X_valid.shape, "Test:", X_test.shape)
print("Object(categorical) features:", cat_features)

Train: (348, 112) Valid: (62, 112) Test: (73, 112)
Object(categorical) features: []


In [6]:
X_train.head()

,year_published,programming,ml_dl_core,llm_nlp,cv_audio_multimodal,ml_domains,frameworks_libraries,data_engineering,mlops_production,cloud_platform,...,region_тула,region_тюмень,region_ульяновск,region_уфа,region_хабаровск,region_чайковский,region_чебоксары,region_челябинск,region_череповец,region_якутск
456,2026,"c++, python",NaN,NaN,computer_vision,NaN,"numpy, opencv, pandas, tensorflow",NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
110,2026,python,NaN,"fine_tuning, llm, nlp",NaN,NaN,NaN,NaN,api_backend,NaN,...,0,0,0,0,0,0,0,0,0,0
149,2026,"python, sql",NaN,llm,NaN,time_series,NaN,NaN,kubernetes,NaN,...,0,0,0,0,0,0,0,0,1,0
423,2026,NaN,NaN,nlp,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
422,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0


In [15]:
model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    verbose=200,
    early_stopping_rounds=100,
    cat_features=cat_features,
 )

model.fit(X_train, y_train, eval_set=(X_valid, y_valid), use_best_model=True)

valid_pred = model.predict(X_valid)
test_pred = model.predict(X_test)


def regression_metrics(y_true, y_pred, prefix=""):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))

CatBoostError: Bad value for num_feature[non_default_doc_idx=0,feature_idx=1]="c++, python": Cannot convert 'c++, python' to float

In [ ]:
# Metrics in RUB

y_test_rub = np.expm1(y_test)
test_pred_rub = np.expm1(test_pred)

rub_rmse = float(np.sqrt(mean_squared_error(y_test_rub, test_pred_rub)))
rub_mae = float(mean_absolute_error(y_test_rub, test_pred_rub))

print("\nTest metrics (RUB):")
print(f"test_RMSE_rub: {rub_rmse:.2f}")
print(f"test_MAE_rub:  {rub_mae:.2f}")

In [ ]:
# Feature importance + saving
feature_importance = pd.DataFrame(
    {"feature": X_train.columns, "importance": model.get_feature_importance()}
).sort_values("importance", ascending=False)

print("\nTop 30 features:")
print(feature_importance.head(30))

model.save_model(CATBOOST_MODEL_OUT)
feature_importance.to_csv(CATBOOST_FI_OUT, index=False)

# Test predictions dataframe
pred_df = X_test.copy()
pred_df["y_true_log"] = y_test.values
pred_df["y_pred_log"] = test_pred
pred_df["y_true_rub"] = np.expm1(y_test).values
pred_df["y_pred_rub"] = np.expm1(test_pred)

pred_df.to_csv(CATBOOST_TEST_PRED_OUT, index=False)

print("\nSaved:")
print("-", CATBOOST_MODEL_OUT)
print("-", CATBOOST_FI_OUT)
print("-", CATBOOST_TEST_PRED_OUT)

In [ ]:
# =========================
# Part B: Text model (TF-IDF + Ridge) with OOF predictions
# =========================
df_text = pd.read_csv(str(TEXT_FILE))

required = ["vacancy_text", "salary"]
missing = [c for c in required if c not in df_text.columns]
if missing:
    raise KeyError(f"Missing columns in {TEXT_FILE}: {missing}")

# Optional filter
if "ml_dl_label" in df_text.columns:
    df_text = df_text[df_text["ml_dl_label"] != "точно нет"].copy()

# salary -> rub
salary_rub = df_text["salary"].apply(parse_salary_to_rub)
df_text["salary_rub"] = salary_rub
df_text = df_text[df_text["salary_rub"].notna()].copy()
df_text = df_text[df_text["salary_rub"] > 0].copy()
df_text["salary_rub_log"] = np.log1p(df_text["salary_rub"])

# text exists
X_raw = df_text["vacancy_text"].astype(str)
y = df_text["salary_rub_log"].astype(float)

# trim extreme outliers
low, high = y.quantile(0.01), y.quantile(0.99)
mask = (y >= low) & (y <= high)
df_text = df_text[mask].copy()

X_raw = df_text["vacancy_text"].astype(str)
y = df_text["salary_rub_log"].astype(float)

print("Rows for text model:", len(df_text))

In [ ]:
# Minimal, model-friendly text normalization (keeps words + underscores)
_RE_URL = re.compile(r"http\S+|www\.\S+", flags=re.IGNORECASE)
_RE_EMAIL = re.compile(r"\S+@\S+", flags=re.IGNORECASE)
_RE_NON_WORD = re.compile(r"[^a-zа-яё0-9_\s]", flags=re.IGNORECASE)
_RE_SPACES = re.compile(r"\s+")


def normalize_text(text: object) -> str:
    s = unicodedata.normalize("NFKC", html.unescape(str(text))).lower()
    s = _RE_URL.sub(" ", s)
    s = _RE_EMAIL.sub(" ", s)
    s = _RE_NON_WORD.sub(" ", s)
    s = _RE_SPACES.sub(" ", s).strip()
    return s


df_text["text_preprocessed"] = X_raw.apply(normalize_text)
df_text = df_text[df_text["text_preprocessed"].str.strip().ne("")].copy()

print("Rows after text preprocessing:", len(df_text))

In [ ]:
# CV + OOF predictions
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=False,  # already normalized
            max_features=70000,
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.95,
            sublinear_tf=True,
        ),
    ),
    ("regressor", Ridge(alpha=3.0)),
])

X = df_text["text_preprocessed"]
y = df_text["salary_rub_log"].astype(float)

oof_pred = np.zeros(len(df_text), dtype=float)
rmse_scores, mae_scores, r2_scores = [], [], []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), 1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    oof_pred[valid_idx] = pred

    rmse = float(np.sqrt(mean_squared_error(y_valid, pred)))
    mae = float(mean_absolute_error(y_valid, pred))
    r2 = float(r2_score(y_valid, pred))

    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)

    print(f"Fold {fold}: RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")

print("\nCV results:")
print(f"RMSE mean = {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
print(f"MAE  mean = {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"R2   mean = {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")

In [ ]:
# Save preprocessed & OOF outputs

df_text.to_csv(TEXT_PREPROCESSED_OUT, index=False)

result_cols = [c for c in ["vacancy_id", "region", "year_published"] if c in df_text.columns]
result_df = df_text[result_cols].copy() if result_cols else pd.DataFrame(index=df_text.index)

result_df["salary_rub"] = df_text["salary_rub"].values
result_df["salary_rub_log"] = df_text["salary_rub_log"].values
result_df["text_preprocessed"] = df_text["text_preprocessed"].values
result_df["pred_salary_rub_log_oof"] = oof_pred
result_df["pred_salary_rub_oof"] = np.expm1(oof_pred)

result_df.to_csv(TEXT_OOF_OUT, index=False)

print("Saved:")
print("-", TEXT_PREPROCESSED_OUT)
print("-", TEXT_OOF_OUT)